# Python script for generating CSS_hate database sql script

css_hate_merge_records.sql

In [1]:
import pandas as pd

df = pd.read_csv("Oncampushate202122.csv")

# Define key columns
primary_key = "OPEID"
first_columns = ["OPEID", "men_total", "women_total", "Total", "UNITID_P", "sector_cd", "FILTER20", "FILTER21", "FILTER22"]  # Columns to take first occurrence
first_occurrence_columns = first_columns[1:4]

# Identify numeric columns to sum (exclude first_columns)
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
sum_columns = [col for col in numeric_columns if col not in first_columns]

# Generate SQL script
sql_script = []

# Create Table
sql_script.append("CREATE TABLE `css`.`css_hate_merged_records` (\n    " +
                  f"{primary_key} INT PRIMARY KEY,\n    " +
                  ",\n    ".join([f"{col} INT" for col in first_occurrence_columns + sum_columns]) + "\n);")

# Insert aggregated data
sql_script.append("\nINSERT INTO `css`.`css_hate_merged_records`\nSELECT \n    " +
                  f"{primary_key},\n    " +
                  ",\n    ".join([f"MIN({col}) AS {col}" for col in first_occurrence_columns]) + ",\n    " +
                  ",\n    ".join([f"SUM({col}) AS {col}" for col in sum_columns]) +
                  f"\nFROM `css`.`css_hate`\nGROUP BY {primary_key};")

# Save to a .sql file
with open("css_hate_merge_records.sql", "w") as f:
    f.write("\n".join(sql_script))

print("SQL script generated successfully: css_hate_merge_records.sql")


SQL script generated successfully: css_hate_merge_records.sql


<ipython-input-1-deaa59e772ef>:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Oncampushate202122.csv")


css_hate_final.sql

In [3]:
# Define first_columns (not summed)
first_columns = ["OPEID", "men_total", "women_total", "Total", "UNITID_P", "sector_cd", "FILTER20", "FILTER21", "FILTER22"]

# Identify numeric columns to sum (exclude first_columns)
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
sum_columns = [col for col in numeric_columns if col not in first_columns]

# Generate SQL script
sql_script = []

# Disable SQL safe mode
sql_script.append("SET SQL_SAFE_UPDATES = 0;")

# Add YEARLYHATECRIME column
sql_script.append("\nALTER TABLE `css`.`css_hate_merged_records`\nADD COLUMN YEARLYHATECRIME DOUBLE;")

# Update YEARLYHATECRIME with computed value
sql_script.append("\nUPDATE `css`.`css_hate_merged_records`\nSET YEARLYHATECRIME = (\n    " +
                  " +\n    ".join(sum_columns) + f"\n) / {len(sum_columns)};")

# Add YEARLYHATECRIME1K column
sql_script.append("\nALTER TABLE `css`.`css_hate_merged_records`\nADD COLUMN YEARLYHATECRIME1K DOUBLE;")

# Update YEARLYHATECRIME1K with computed value
sql_script.append("\nUPDATE `css`.`css_hate_merged_records`\nSET YEARLYHATECRIME1K = YEARLYHATECRIME * 1000 / NULLIF(Total, 0);")

# Step 5: Rename table
sql_script.append("\nRENAME TABLE `css`.`css_hate_merged_records` TO `css`.`css_hate_final`;")

# Enable SQL safe mode
sql_script.append("\nSET SQL_SAFE_UPDATES = 1;")

# Save to a .sql file
with open("css_hate_final.sql", "w") as f:
    f.write("\n".join(sql_script))

print("SQL script generated successfully: css_hate_final.sql")

SQL script generated successfully: css_hate_final.sql
